In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class LightGCNEncoder(nn.Module):
    def __init__(self, num_users, num_items, embedding_dim=64, num_layers=3):
        super(LightGCNEncoder, self).__init__()
        self.num_users = num_users
        self.num_items = num_items
        self.embedding_dim = embedding_dim
        self.num_layers = num_layers

        # initial embeddings
        self.user_embedding = nn.Embedding(num_users, embedding_dim)
        self.item_embedding = nn.Embedding(num_items, embedding_dim)
        nn.init.normal_(self.user_embedding.weight, std=0.1)
        nn.init.normal_(self.item_embedding.weight, std=0.1)

    def forward(self, norm_adj):
        """
        inputs:
            norm_adj = normalized adjacency matrix
        outputs:
            final_user_embeddings
            final_item_embeddings
        """
        # layer 0 embddings
        users_emb = self.user_embedding.weight
        items_emb = self.item_embedding.weight
        
        # concat of user and items embeddings to make complete matrix
        all_emb = torch.cat([users_emb, items_emb], dim=0)
        embs = [all_emb]

        # message passing
        for layer in range(self.num_layers):
            all_emb = torch.sparse.mm(norm_adj, all_emb)
            embs.append(all_emb)

        # layer combination
        embs = torch.stack(embs, dim=1)
        final_emb = torch.mean(embs, dim=1)

        # splitting final embeddings into user embeddings and item embeddings
        final_user_embeddings, final_item_embeddings = torch.split(
            final_emb, [self.num_users, self.num_items]
        )

        return final_user_embeddings, final_item_embeddings

In [4]:
class BehaviorPrototypeLearning(nn.Module):
    """
    Differentiable behavior prototype learning module.

    Learns separate behavior prototypes for users and items,
    computes soft prototype assignments, and builds
    prototype-aware representations.
    """

    def __init__(
        self,
        embedding_dim=64,
        num_prototypes=5,
        temperature=1.0
    ):
        super(BehaviorPrototypeLearning, self).__init__()
        self.embedding_dim = embedding_dim
        self.num_prototypes = num_prototypes
        self.temperature = temperature
        # Differentiable Behavior Prototype Learning
        self.user_prototypes = nn.Parameter(
            torch.randn(num_prototypes, embedding_dim)
        )
        self.item_prototypes = nn.Parameter(
            torch.randn(num_prototypes, embedding_dim)
        )
        # Prototype Representation  (Wp​ in pape)
        self.user_projection = nn.Linear(
            embedding_dim,
            embedding_dim
        )
        self.item_projection = nn.Linear(
            embedding_dim,
            embedding_dim
        )

    def forward_user(self, embeddings):
        """Compute user prototype assignments and representations."""

        # Distance between users and prototypes
        distances = torch.sum(
            (
                embeddings.unsqueeze(1)
                - self.user_prototypes.unsqueeze(0)
            ) ** 2,
            dim=2
        )
        # Soft prototype assignment
        assignments = F.softmax(
            -distances / self.temperature,
            dim=1
        )
        # Prototype Representation
        prototype_representation = torch.matmul(
            assignments,
            self.user_prototypes
        )
        prototype_representation = self.user_projection(
            prototype_representation
        )
        prototype_representation = torch.sigmoid(
            prototype_representation
        )
        return assignments, distances, prototype_representation

    def forward_item(self, embeddings):
        """Compute item prototype assignments and representations."""

        # Distance between items and prototypes
        distances = torch.sum(
            (
                embeddings.unsqueeze(1)
                - self.item_prototypes.unsqueeze(0)
            ) ** 2,
            dim=2
        )
        # Soft prototype assignment
        assignments = F.softmax(
            -distances / self.temperature,
            dim=1
        )
        # Prototype Representation
        prototype_representation = torch.matmul(
            assignments,
            self.item_prototypes
        )
        prototype_representation = self.item_projection(
            prototype_representation
        )
        prototype_representation = torch.sigmoid(
            prototype_representation
        )

        return assignments, distances, prototype_representation

    def user_prototype_loss(
        self,
        embeddings,
        assignments,
        distances
    ):
        """Compute the user prototype learning loss."""

        loss = torch.sum(
            assignments * distances
        )
        return loss
    
    def item_prototype_loss(
        self,
        embeddings,
        assignments,
        distances
    ):
        """Compute the item prototype learning loss."""
        loss = torch.sum(
            assignments * distances
        )
        return loss